## Patch manipulations basics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from yeti_iga.future.bspline import (BSpline, BSplineSurface, ControlPointManager,
    Patch, HRefiner, SubdivisionRefiner, PRefiner, BezierExtractor
)
from yeti_iga.future.plotting import plot_patches_2d

## Create a 2D patch

A B-spline patch in `yeti_iga.future` is built from three building blocks:

1. **`ControlPointManager`** — manages a shared pool of control points in physical space.
   Every point is stored once; multiple patches can reference the same pool.
2. **`BSplineSurface`** / **`BSplineVolume`** — describes the parametric space via one
   `BSpline(p, kv)` object per direction (degree `p`, knot vector `kv`).
3. **`Patch`** — links a parametric space to a set of control points through a `mapping`
   array and a `local_shape`.

In [ ]:
# Create a set of control points
mgr = ControlPointManager(dim=2)
mgr.add_point([0.0, 1.0])
mgr.add_point([1.0, 0.0])
mgr.add_point([0.0, 2.0])
mgr.add_point([2.0, 0.0])
mgr.add_point([1.0, 1.0])
mgr.add_point([2.0, 2.0])

# Plot points with their indices in CP manager
def plot_control_points(coords):
    fig, ax = plt.subplots()

    ax.scatter(coords[:, 0], coords[:, 1], zorder=3)
    for i, (x, y) in enumerate(coords):
        ax.annotate(str(i), (x, y), textcoords="offset points", xytext=(6, 6))

    ax.set_aspect('equal')
    ax.grid(True)
    plt.show()

coords = mgr.coords_view()  # shape (n_points, 2)
plot_control_points(coords)


### Parametric space and `Patch`

`BSpline(p, kv)` encodes a 1D B-spline basis:
- `p` — polynomial degree
- `kv` — open (clamped) knot vector of length $n + p + 1$; the number of basis functions is $n = |kv| - p - 1$

`local_shape = [n_u, n_v, ...]` is the shape of the logical CP grid in the patch.
The `mapping` list converts a **u-fastest flat index** `flat = iu + iv*n_u + ...`
into a global CP-manager index.

In [ ]:
# Create a 2D BSpline parametric space
# (degree 2 in direction 1, degree 1 in direction 2)
su = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))
sv = BSpline(1, np.array([0., 0., 1., 1.]))
surf = BSplineSurface(su, sv)

# Set mapping to control points
mapping = np.array([0, 4, 1, 2, 5, 3], dtype = np.int64)
# Number of functions in each parametric direction
local_shape = [3, 2]

# Create patch
patch = Patch(surf, mgr, mapping.tolist(), local_shape)


## Evaluate patch

Evaluating the B-spline map $\mathbf{x}(\xi)$ at a parameter point $\xi$ requires
two steps:

1. **`find_span(xi)`** — locates the knot span $k$ such that $U_k \le \xi < U_{k+1}$
2. **`evaluate_patch_nd(spans, params)`** — computes
   $\mathbf{x}(\xi) = \sum_i N_i(\xi)\, P_i$
   using only the $\prod_d (p_d+1)$ basis functions active in the span

`evaluate_patch_nd_omp` is the OpenMP-parallelised version for batched evaluation.

In [ ]:
# Evaluate 10 points at several points
n_points = 10
u = np.random.rand(n_points, 2)

spans = np.array([surf.find_span_nd(pt) for pt in u])
# Serial evaluation of point
res_serial = patch.evaluate_patch_nd(spans, u)
# Parallel evaluation of points
res_omp = patch.evaluate_patch_nd_omp(spans, u)

for pt_param, pt_span, pt_serial, pt_omp in zip(u, spans, res_serial, res_omp):
    print(f'{pt_param}, span: {pt_span}, serial: {pt_serial}, OMP: {pt_omp}')

In [ ]:
# Plot patch (iso-parametric element borders, control points and their global ids)
plot_patches_2d(patch, show_control_points=True, show_control_point_indices=True,
                 title='Initial patch')

## Refinement

Three refinement strategies are available, all returning a **transition matrix** $T$
such that the new control-point vector satisfies $\hat{P} = T \cdot P$.
The patch geometry is preserved exactly; only the representation changes.

| Refiner | Strategy | Effect |
|---|---|---|
| `PRefiner(direction, n_elevations)` | degree elevation | degree $+n$, new CPs inserted |
| `HRefiner(direction, knot)` | knot insertion | one new knot, $p$ new CPs |
| `SubdivisionRefiner(direction, n_levels)` | bisection | $2^{n_\text{levels}}\times$ more elements |

### p-refinement — degree elevation

`PRefiner(direction, n_elevations)` raises the polynomial degree by `n_elevations`
in one parametric direction.  The number of elements is unchanged; new control points
are added to maintain $C^{p-1}$ continuity across every existing knot span.

In [ ]:
# Degree elevation
T = PRefiner(direction=1, n_elevations=1).refine(patch)

# Visu
plot_patches_2d(patch, show_control_points=True, show_control_point_indices=True,
                 title='After degree elevation')

### h-refinement — knot insertion

`HRefiner(direction, knot)` inserts a single knot value into the knot vector of
direction `d`.  The geometry is preserved.

In [ ]:
# Knot insertion in 1st parametric direction
T = HRefiner(direction=0, knot=0.33).refine(patch)

# Visu
plot_patches_2d(patch, show_control_points=True, show_control_point_indices=True,
                 title='After knot insertion (direction 0)')

In [ ]:
# Knot insertion in 2nd parametric direction
T = HRefiner(direction=1, knot=0.66).refine(patch)

# Visu
plot_patches_2d(patch, show_control_points=True, show_control_point_indices=True,
                 title='After knot insertion (direction 1)')

### Subdivision refinement

`SubdivisionRefiner(direction, n_levels)` bisects every existing knot span
`n_levels` times, multiplying the element count by $2^{n_\text{levels}}$.
This is equivalent to repeated `HRefiner` calls at each midpoint.

In [ ]:
# Subdivide span in both directions
T = SubdivisionRefiner(direction=0, n_levels=2).refine(patch)
T = SubdivisionRefiner(direction=1, n_levels=2).refine(patch)

# Visu
plot_patches_2d(patch, show_control_points=True, show_control_point_indices=True,
                 title='After subdivision')

print(f'Patch have {mgr.n_points} control points')

---
## Bézier extraction

Bézier extraction (Borden et al. 2011) decomposes a B-spline patch into independent
Bernstein elements.

In [ ]:

# Run Bézier extraction
elems = BezierExtractor.extract_nd(patch)


for elem in elems:
    P_active = np.array([patch.control_point(j) for j in elem.active_indices])
    P_bz = elem.C.T @ P_active   # Bézier CPs: P_bz = (C^e)^T @ P_active
    print(f"Element {tuple(elem.elem_index)}  |  active indices: {list(elem.active_indices)}")
    print(f"  Extraction operator C^e:\n{np.round(elem.C, 3)}")
    print(f"  Bézier CPs (u-fastest):\n{np.round(P_bz, 4)}\n")

In [ ]:
def plot_bezier_elements_2d(patch, title='Bézier extraction'):
    """Plot Bézier control polygons (one colour per element)."""
    elems = BezierExtractor.extract_nd(patch)
    su = patch.tensor.components[0]
    sv = patch.tensor.components[1]
    pu, pv = su.degree, sv.degree
    u_breaks = np.unique(su.knot_vector)
    v_breaks = np.unique(sv.knot_vector)
    n_elems = len(elems)
    colors = plt.cm.tab10(np.arange(n_elems) % 10)

    fig, ax1 = plt.subplots(1, 1, figsize=(6.5, 5))

    # Iso-parametric lines + Bézier control polygons
    ax1.set_title(f'{title}  ({n_elems} element(s), degree {pu}×{pv})')
    n_samp = 80
    for u_val in u_breaks:
        vs = np.linspace(v_breaks[0], v_breaks[-1], n_samp)
        spans = np.array([[su.find_span(u_val), sv.find_span(v)] for v in vs], dtype=np.int32)
        pts = patch.evaluate_patch_nd_omp(spans,
                np.column_stack([np.full(n_samp, u_val), vs]))
        ax1.plot(pts[:, 0], pts[:, 1], 'b-', lw=1.2)
    for v_val in v_breaks:
        us = np.linspace(u_breaks[0], u_breaks[-1], n_samp)
        spans = np.array([[su.find_span(u), sv.find_span(v_val)] for u in us], dtype=np.int32)
        pts = patch.evaluate_patch_nd_omp(spans,
                np.column_stack([us, np.full(n_samp, v_val)]))
        ax1.plot(pts[:, 0], pts[:, 1], 'b-', lw=1.2)

    for elem, color in zip(elems, colors):
        P_active = np.array([patch.control_point(j) for j in elem.active_indices])
        P_bz = elem.C.T @ P_active                     # (C^e)^T @ P_active
        P_grid = P_bz.reshape(pv + 1, pu + 1, -1)     # (nv_loc, nu_loc, dim)
        for iv in range(pv + 1):
            ax1.plot(P_grid[iv, :, 0], P_grid[iv, :, 1], '--', color=color, lw=1.0)
        for iu in range(pu + 1):
            ax1.plot(P_grid[:, iu, 0], P_grid[:, iu, 1], '--', color=color, lw=1.0)
        ax1.scatter(P_bz[:, 0], P_bz[:, 1], s=40, c=[color], zorder=3)
        ax1.annotate(f'e{tuple(elem.elem_index)}', P_bz.mean(axis=0),
                     ha='center', va='center', fontsize=9,
                     color=color, fontweight='bold')

    ax1.set_aspect('equal')
    ax1.grid(True)
    plt.tight_layout()
    plt.show()

plot_bezier_elements_2d(patch)

---
## VTU export (Paraview)

`write_bezier_patch_vtu` writes each B-spline element as a high-order VTK Bézier cell:
- 2D patch → **`VTK_BEZIER_QUADRILATERAL`** (cell type 77)
- 3D patch → **`VTK_BEZIER_HEXAHEDRON`** (cell type 79)

Bézier control points per element are computed as $P^e_\text{Bézier} = (C^e)^\top P^e_\text{active}$.
A scalar or vector field defined at the B-spline CPs is transformed the same way,
so Paraview interpolates it correctly inside each curved element.

> **Requirement**: Paraview 5.9+ / VTK 9.0+ for correct high-order Bézier rendering.

In [ ]:
from yeti_iga.future.vtu import write_bezier_patch_vtu
import os

os.makedirs('output', exist_ok=True)

# Geometry only
write_bezier_patch_vtu(patch, 'output/patch.vtu')
print("Geometry exported → output/patch.vtu")

# With a scalar field: distance from each CP to the geometric centre
all_cp = np.array([patch.control_point(i) for i in range(patch.n_cp)])
center = all_cp.mean(axis=0)
center = np.array([0.0, 0.0])
field_dist = np.linalg.norm(all_cp - center, axis=1)

write_bezier_patch_vtu(patch, 'output/patch_field.vtu',
                        field=field_dist, field_name='dist_to_centre')
print("With scalar field   → output/patch_field.vtu")

print("\nOpen in Paraview:")
print("  File > Open > output/patch_field.vtu")
print("  Apply  →  patch rendered with curved Bézier edges")
print("  Colour by 'dist_to_centre'")